In [ ]:
import tonic
import tonic.transforms as T
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm

batch_size = 2
epochs = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
gesture_map = {
    0: "hand_clapping",
    1: "right_hand_wave",
    2: "left_hand_wave",
    3: "right_hand_clockwise",
    4: "right_hand_counter_clockwise",
    5: "left_hand_clockwise",
    6: "left_hand_counter_clockwise",
    7: "forearm_roll_forward",
    8: "forearm_roll_backward",
    9: "drums",
    10: "guitar",
    #11: "random_other_gestures", # -> not implemented here
}

In [ ]:
sensor_size = tonic.datasets.DVSGesture.sensor_size

transform = T.Compose([
    T.ToFrame(sensor_size=sensor_size, time_window=50 * 10 ** 3)
])

train_dataset = tonic.datasets.DVSGesture(save_to="./data", transform=transform, train=True)
test_dataset = tonic.datasets.DVSGesture(save_to="./data", transform=transform, train=False)
# we must pad the frames to the same size

# frames shape is (time_bins, num_polarities, height, width)
# dataloader will return (batch_size, time_bins, num_polarities, height, width)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=tonic.collation.PadTensors(), num_workers=0, pin_memory=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=tonic.collation.PadTensors(), num_workers=0, pin_memory=False)
firing_threshold = 1.0
model = basicSNN(num_classes=len(gesture_map), num_channels=2, threshold=firing_threshold).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0
    epoch_test_loss = 0
    for event_batches, labels in tqdm(train_loader, total=len(train_loader)):
        event_batches = event_batches.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        out_spikes, out_mem = model(event_batches)
        raw_mem = compute_raw_mem(out_spikes, out_mem, threshold=firing_threshold)
        # print(event_batches.shape)
        # print(out_spikes.shape) # b, t, num_classes
        out_spikes = out_spikes.sum(dim=1) # sum over time
        # print(f"out spikes are {out_spikes} correct labels are {labels}")
        # print(out_spikes.shape)
        # print(labels.shape)
        out_classes = F.softmax(out_spikes, dim=1)
        loss = criterion(out_spikes.view(-1, len(gesture_map)), labels.view(-1))
        loss.backward()
        epoch_train_loss += loss.item()
        optimizer.step()
    
    print(f"Epoch {epoch}, Train Loss: {epoch_train_loss / len(train_loader)}")
    
    # model.eval()
    # # Testing
    # with torch.no_grad():
    #     for event_batches, labels in tqdm(test_loader, total=len(test_loader)):
    #         event_batches = event_batches.to(device)
    #         labels = labels.to(device)
    #         out_spikes = model(event_batches)
    #         out_spikes = out_spikes.mean(dim=1) # average over time
    #         # print(out_spikes.shape)
    #         loss = criterion(out_spikes.view(-1, len(gesture_map)), labels.view(-1))
    #         epoch_test_loss += loss.item()

    # print(f"Epoch {epoch}, Test Loss: {epoch_test_loss / len(test_loader)}")
